# ⚖️ Conciliação Unificada — Motor v5

Busque por **Cidade**, **CNPJ** ou **Nome** + Valor Alvo.
O solver MILP testa 4 variantes por nota (Saldo, Saldo−IR, Saldo−ISS, Líquido).

> 🟢 **██** = célula usada na composição &nbsp;|&nbsp; Espaço vazio = não usado

In [ ]:
!pip install pandas openpyxl scipy -q

In [ ]:
import pandas as pd
import numpy as np
import time, io, re
from scipy.optimize import milp, LinearConstraint, Bounds

TOLERANCIA = 0.01
MAX_NOTAS = 30

COLUNAS = {
    "nome":["NOME CLIENTE","NOME"], "cnpj":["CNPJ"], "cidade":["CIDADE"], "uf":["UF"],
    "nf":["NF","NR NFEM"], "num_doc":["DOC","NUM DOC MATERA"],
    "rbase":["RBASE RAIZ","RBASE"], "valor_titulo":["BRUTO","VLR TITULO","VALOR TITULO"],
    "saldo":["SALDO","VLR SALDO","VALOR SALDO"], "ir":["IR","VLR RETIDO"],
    "iss":["ISS","ISS RETIDO"], "venc":["VENCIMENTO","VENC","DT VENCIMENTO"],
    "atraso":["ATRASO"],
}

def carregar_planilha(path_or_bytes):
    if isinstance(path_or_bytes, str):
        df = pd.read_excel(path_or_bytes)
    else:
        df = pd.read_excel(io.BytesIO(path_or_bytes))
    df.columns = df.columns.str.strip()
    cols_upper = {c.strip().upper(): c for c in df.columns}
    renomear = {}
    for destino, candidates in COLUNAS.items():
        for cand in candidates:
            if cand.upper() in cols_upper and destino not in renomear.values():
                renomear[cols_upper[cand.upper()]] = destino; break
    df.rename(columns=renomear, inplace=True)
    for col in ["valor_titulo","saldo","ir","iss","atraso"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).round(2)
    if "cnpj" in df.columns:
        df["cnpj_limpo"] = df["cnpj"].astype(str).str.replace(r"\D","",regex=True)
    df.index = range(len(df))
    return df

def gerar_opcoes_nota(row):
    s = round(float(row.get("saldo",0)),2)
    ir = round(float(row.get("ir",0)),2)
    iss = round(float(row.get("iss",0)),2)
    ops = [("saldo", s)]
    if ir != 0: ops.append(("saldo_ir", round(s+ir,2)))
    if iss != 0: ops.append(("saldo_iss", round(s+iss,2)))
    if ir != 0 and iss != 0:
        ops.append(("liquido", round(s+ir+iss,2)))
    seen, deduped = set(), []
    for nome, val in ops:
        if val > 0 and val not in seen: seen.add(val); deduped.append((nome, val))
    return deduped

def solver_milp(df_notas, alvo, max_notas=MAX_NOTAS):
    keys, vals = [], []
    for idx, row in df_notas.iterrows():
        for var, val in gerar_opcoes_nota(row):
            if val > 0 and val <= alvo + TOLERANCIA:
                keys.append((idx, var)); vals.append(val)
    n = len(vals)
    if n == 0: return [], 'Nenhum valor positivo'
    vals_arr = np.array(vals)
    note_groups = {}
    for i, (idx, var) in enumerate(keys): note_groups.setdefault(idx, []).append(i)
    constraints = [LinearConstraint(vals_arr.reshape(1,-1), lb=alvo-TOLERANCIA, ub=alvo+TOLERANCIA)]
    constraints.append(LinearConstraint(np.ones((1,n)), lb=0, ub=max_notas))
    for indices in note_groups.values():
        if len(indices) > 1:
            row = np.zeros(n)
            for i in indices: row[i] = 1
            constraints.append(LinearConstraint(row.reshape(1,-1), lb=0, ub=1))
    bounds = Bounds(lb=np.zeros(n), ub=np.ones(n))
    t0 = time.time()
    res = milp(np.ones(n), constraints=constraints, integrality=np.ones(n), bounds=bounds)
    elapsed = time.time()-t0
    if res.success and res.x is not None:
        xi = np.round(res.x).astype(int)
        selected = [(keys[i][0], keys[i][1], vals[i]) for i in range(n) if xi[i]==1]
        soma = round(sum(v for _,_,v in selected),2)
        return selected, f'✅ {len(selected)} notas, soma=R$ {soma:,.2f}, Δ=R$ {abs(soma-alvo):,.2f} ({elapsed:.1f}s)'
    return [], f'❌ {res.message} ({elapsed:.1f}s)'

TIPO_LABEL = {'saldo':'🟢 Saldo','saldo_ir':'🟡 Saldo−IR','saldo_iss':'🟡 Saldo−ISS','liquido':'🔴 Líquido'}
TIPO_COLS  = {'saldo':{'saldo'},'saldo_ir':{'saldo','ir'},'saldo_iss':{'saldo','iss'},'liquido':{'saldo','ir','iss'}}

def brl(v):
    try: return f'R$ {float(v):,.2f}'.replace(',','X').replace('.',',').replace('X','.')
    except: return '—'

def exibir(sol, df_cli, alvo, escopo):
    print(f'\n{"─"*70}')
    print(f'  ✅ COMPOSIÇÃO — {escopo}')
    print(f'{"─"*70}')
    linhas = []
    for idx, variante, val in sol:
        row = df_cli.loc[idx]
        s,ir_v,iss_v = round(float(row.get('saldo',0)),2), round(float(row.get('ir',0)),2), round(float(row.get('iss',0)),2)
        cv = TIPO_COLS.get(variante, set())
        ms = '██' if 'saldo' in cv else '  '
        mi = '██' if 'ir' in cv else '  '
        mi2= '██' if 'iss' in cv else '  '
        print(f'  NF {str(row.get("nf","")):>10} │ {ms} Saldo {s:>12,.2f} │ {mi} IR {ir_v:>8,.2f} │ {mi2} ISS {iss_v:>8,.2f} │ → {val:>12,.2f} │ {TIPO_LABEL.get(variante,variante)}')
        linhas.append({'NF':row.get('nf',''),'Nome':str(row.get('nome',''))[:40],'Saldo':s,'IR':ir_v,'ISS':iss_v,'Vlr Usado':val,'Tipo':TIPO_LABEL.get(variante,variante)})
    soma = round(sum(v for _,_,v in sol),2)
    print(f'{"─"*70}')
    print(f'  TOTAL: {soma:>12,.2f}  │  Alvo: {alvo:>12,.2f}  │  Δ: {abs(soma-alvo):,.2f}')
    print(f'  {len(sol)} nota(s)')
    print(f'{"─"*70}')
    return pd.DataFrame(linhas)

def conciliar(df, filtro_tipo, filtro_valor, valor_alvo):
    ft = filtro_tipo.upper()
    if ft == 'CIDADE':
        df_cli = df[df['cidade'].str.strip().str.upper() == filtro_valor.strip().upper()]
    elif ft == 'CNPJ':
        limpo = re.sub(r'\D','',filtro_valor).lstrip('0')
        col_c = 'cnpj_limpo' if 'cnpj_limpo' in df.columns else 'cnpj'
        df_cli = df[df[col_c].str.lstrip('0') == limpo]
    else:
        df_cli = df[df['nome'].str.strip().str.upper().str.contains(filtro_valor.strip().upper(), na=False)]
    if len(df_cli)==0: print(f'❌ Nenhum registro para {filtro_tipo}: {filtro_valor}'); return None
    print(f'\n{"="*70}')
    print(f'  {filtro_tipo}: {filtro_valor} → {brl(valor_alvo)}')
    print(f'  {len(df_cli)} notas encontradas')
    print(f'{"="*70}')
    col_cnpj = 'cnpj_limpo' if 'cnpj_limpo' in df_cli.columns else 'cnpj'
    # Camada A: CNPJ individual
    if ft == 'CIDADE':
        print(f'\n🔍 Camada A: CNPJ individual...')
        for cv in df_cli[col_cnpj].unique():
            ds = df_cli[df_cli[col_cnpj]==cv]
            nome = ds['nome'].iloc[0] if 'nome' in ds.columns else cv
            print(f'  CNPJ {cv} ({nome}): {len(ds)} notas')
            sol, msg = solver_milp(ds, valor_alvo)
            print(f'  {msg}')
            if sol: return exibir(sol, df_cli, valor_alvo, f'CNPJ {cv}')
        # Camada B: RBASE
        if 'rbase' in df_cli.columns:
            print(f'\n🔍 Camada B: Grupo RBASE...')
            for rv in df_cli['rbase'].unique():
                ds = df_cli[df_cli['rbase']==rv]
                if ds[col_cnpj].nunique() < 2: continue
                print(f'  RBASE {rv}: {len(ds)} notas')
                sol, msg = solver_milp(ds, valor_alvo)
                print(f'  {msg}')
                if sol: return exibir(sol, df_cli, valor_alvo, f'RBASE {rv}')
        # Camada C: cidade completa
        print(f'\n🔍 Camada C: Cidade completa ({len(df_cli)} notas)...')
    else:
        print(f'\n🔍 Buscando composição...')
    sol, msg = solver_milp(df_cli, valor_alvo)
    print(f'  {msg}')
    if sol: return exibir(sol, df_cli, valor_alvo, f'{filtro_tipo}: {filtro_valor}')
    print(f'\n❌ Nenhuma combinação encontrada.')
    return None

print('✅ Motor carregado')


## 📂 Upload da planilha de títulos

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = carregar_planilha(uploaded[file_name])
print(f'\n✅ {len(df):,} títulos carregados')
if 'cidade' in df.columns:
    print(f'   {df["cidade"].str.strip().str.upper().nunique()} cidades')


## 🔍 Conciliar
Edite os campos abaixo e execute:

In [ ]:
#@title Parâmetros { run: "auto" }
BUSCAR_POR = 'Cidade' #@param ['Cidade', 'CNPJ', 'Nome']
IDENTIFICADOR = 'Anitapolis' #@param {type:"string"}
VALOR_ALVO = '279.674,56' #@param {type:"string"}

valor = float(VALOR_ALVO.replace('.','').replace(',','.'))
df_result = conciliar(df, BUSCAR_POR, IDENTIFICADOR, valor)


## 📊 Tabela formatada

In [ ]:
if df_result is not None:
    display(df_result.style.format({'Saldo':'R$ {:,.2f}','IR':'R$ {:,.2f}','ISS':'R$ {:,.2f}','Vlr Usado':'R$ {:,.2f}'}).set_properties(**{'text-align':'right'}, subset=['Saldo','IR','ISS','Vlr Usado']))
    print(f'\nTotal: {brl(df_result["Vlr Usado"].sum())}')


## 💾 Exportar

In [ ]:
if df_result is not None:
    nome_arq = f'conciliacao_{IDENTIFICADOR.replace(" ","_")}.xlsx'
    df_result.to_excel(nome_arq, index=False)
    files.download(nome_arq)
    print(f'✅ {nome_arq}')
